# Setup and genome prep

This notebook sets up dependencies, loads the gene list, and prepares the hg38 reference for sequence extraction.

In [ ]:
# Colab setup (optional)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo_dir = Path(os.environ.get("ENFORMER_REPO", "/content/drive/MyDrive/Enformer Final Project"))
    if repo_dir.exists():
        os.chdir(repo_dir)
        if str(repo_dir) not in sys.path:
            sys.path.insert(0, str(repo_dir))
    else:
        print(f"Repo dir not found: {repo_dir}")
else:
    repo_dir = Path.cwd()

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print(f"Working directory: {Path.cwd()}")


In [ ]:
# Optional: upload files into Colab (hg38.fa.gz, CSVs, PDFs)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
# GPU check
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch not ready:", exc)


In [ ]:
# Optional: download hg38.fa.gz if missing
from pathlib import Path
import urllib.request

HG38_URL = "http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"
hg38_path = Path("hg38.fa.gz")

if not hg38_path.exists():
    print(f"Downloading {HG38_URL} ...")
    urllib.request.urlretrieve(HG38_URL, hg38_path)
else:
    print(f"Found {hg38_path}")


In [ ]:
# Install dependencies (run once)
# !pip install -q enformer-pytorch pyfaidx matplotlib pandas requests


In [ ]:
from pathlib import Path

from src.genome import HG38_FASTA_GZ, ensure_uncompressed_fasta, get_fasta_reader, load_gene_symbols

csv_path = Path("enformer_project_gene_list.csv")

genes = load_gene_symbols(csv_path)
print(f"Loaded {len(genes)} genes. Example: {genes[:5]}")


In [ ]:
# Prepare hg38 FASTA (this is large and can take a while)
# This uses hg38.fa.gz as the source and writes hg38.fa for random access.

fasta_path = ensure_uncompressed_fasta(HG38_FASTA_GZ)
print(f"FASTA ready at: {fasta_path}")

fasta = get_fasta_reader(fasta_path)
print(f"Loaded FASTA with {len(fasta.keys())} contigs")


In [ ]:
# Quick sanity check: fetch a short sequence slice
chrom = "chr1"
start, end = 1_000_000, 1_000_100
seq = fasta[chrom][start:end]
print(seq[:50], len(seq))
